# einops-repeat-broadcast — ex3: few-shot prototype broadcast for cosine-similarity classifier

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat-broadcast`. Running the final beacon cell reports progress against the `Einops: Repeat-as-broadcast` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat-as-broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat-broadcast`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat-broadcast"
DD_SUBTOPIC = "Einops: Repeat-as-broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.repeat as broadcast — quick refresher

`einops.repeat(x, 'a b -> a n b', n=N)` inserts a NEW axis of length `N` *without* allocating `N` copies — internally the new axis is a stride-0 view. This lets you 'pair every X with every Y' (broadcast to `(NX, NY, ...)`) without quadratic memory. The downstream elementwise op materialises the result lazily.

### Exercise 3 — few-shot prototype broadcast for cosine-similarity classifier

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Compose two `einops.repeat` broadcasts (queries vs prototypes) to produce an `(N, C)` cosine-similarity logit matrix without materialising the `(N, C, D)` intermediate.
> Keywords: few-shot, prototype, cosine-similarity, metric-learning, integrative
> ```

**KCs targeted:** `repeat-inserts-zero-stride-axis`, `repeat-pair-every-with-every`

ex1 paired rays × triangles. ex2 broadcast a per-token bias. This one is the *metric-learning* facet: pair every query with every class prototype to get a cosine-similarity logit matrix — the prototypical-net forward pass.

Implement `ex3_proto_logits(queries, prototypes)`:

1. `queries` has shape `(N, D)` — N query embeddings.
2. `prototypes` has shape `(C, D)` — one mean-embedding per class.
3. Use `einops.repeat` to expand:
   - `queries`  → `(N, C, D)` with `'n d -> n c d'`, `c=C`
   - `prototypes` → `(N, C, D)` with `'c d -> n c d'`, `n=N`
4. Compute cosine similarity element-wise along `D`:
   `cos(a, b) = (a · b) / (||a|| ||b||)`
5. Return the `(N, C)` similarity matrix.

Output dtype: `float32`. Values in `[-1, 1]`. The dot product collapses the `D` axis — the broadcast machinery supplies the `(N, C)` shape.

In [ ]:
def ex3_proto_logits(queries: Tensor, prototypes: Tensor) -> Tensor:
    N, D = queries.shape
    C, _ = prototypes.shape
    q = repeat(queries,    'n d -> n c d', c=C)   # (N, C, D), stride-0 along c
    p = repeat(prototypes, 'c d -> n c d', n=N)   # (N, C, D), stride-0 along n
    dots = (q * p).sum(dim=-1)                    # (N, C)
    q_norm = q.norm(dim=-1)                       # (N, C)
    p_norm = p.norm(dim=-1)                       # (N, C)
    return (dots / (q_norm * p_norm + 1e-12)).to(t.float32)


<details><summary>Solution</summary>

```python
def ex3_proto_logits(queries: Tensor, prototypes: Tensor) -> Tensor:
    N, D = queries.shape
    C, _ = prototypes.shape
    q = repeat(queries,    'n d -> n c d', c=C)   # (N, C, D), stride-0 along c
    p = repeat(prototypes, 'c d -> n c d', n=N)   # (N, C, D), stride-0 along n
    dots = (q * p).sum(dim=-1)                    # (N, C)
    q_norm = q.norm(dim=-1)                       # (N, C)
    p_norm = p.norm(dim=-1)                       # (N, C)
    return (dots / (q_norm * p_norm + 1e-12)).to(t.float32)
```

**Two `repeat`s collapse to one shape.** Each query needs a copy per class; each prototype needs a copy per query. `einops.repeat` inserts the missing axis as a stride-0 view, so the `(N, C, D)` intermediate is virtual — no `N*C*D` memory cost until the elementwise op fires.

**Why we don't pre-normalise.** You CAN normalise `queries` and `prototypes` to unit length first and skip the division, and that's what real ProtoNet code does. Here we compute the full formula to make the broadcast pattern visible — the norms are taken AFTER the repeat, on the shared `(N, C, D)` shape, so they too rely on the broadcast.

**Contrast with ex1.** ex1 used `repeat` to pair rays × triangles in a geometry context. This drill is the same broadcast pattern in a metric-learning context — the atom (zero-stride pairing) generalises across domains.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()